# Lab 4.4 · Least Squares as Orthogonal Projection

Find the closest reachable output to b, explain why the residual is perpendicular to Col(A), and distinguish unique fitted values from possibly non-unique coefficients.

Wooseok Ha, MAS110 Lecture 4: Orthogonality and Projections (Fall 2026), lec04.pdf. Least squares: printed slides 61–68 / PDF pages 71–78. Projection: slides 29–33 and 39 / PDF pages 33–37 and 44. QR example: slides 49–51 / PDF pages 57–59.

Maintained in [MAS110 Practice Sessions](https://github.com/statchan1106/mas110-practice-sessions). Course companion to [Foundations of LADS](https://github.com/kyunghyuncho/Foundations_of_LADS) by Wanmo Kang and Kyunghyun Cho.

Run the cells from top to bottom. Only NumPy is required (available in Colab). The normal-equation and explicit QR calculations explain this lecture example; use lstsq directly on A for general fitting.

Chapter 3 asked whether $A\theta  = b$ has an exact solution. Here b may lie outside Col(A), so we choose coefficients whose output is as close to b as possible. Throughout this lab, vectors and matrices are real and distance means the standard Euclidean distance. The running example is exactly the lecture’s 3 × 2 matrix, not a square system with an inverse.

## Notation

- **φ; θ; qⱼ; wⱼ; G**: Across Chapter 4, $\varphi$ denotes angles and $\theta$ denotes model coefficients. In 4.3, $q_{j}$ names a normalized column while $w_{j}$ is a remainder; the lecture previously labels normalized Gram–Schmidt vectors $w_{i}$. G always denotes a Gram matrix: here $A^{\mathsf T}A$, and in 4.2 the polynomial integral metric. The inner product being used determines which Gram matrix applies.

- **A ∈ ℝᵐˣⁿ; here m = 3, n = 2**: A is the design matrix: m measurements (rows) and n coefficients (columns). $\theta  \in  \mathbb R^{n}$ is an input; $A\theta$ and $b \in  \mathbb R^{m}$ are outputs in measurement space. More rows than columns does not itself prove inconsistency; $b \notin  \operatorname{Col}(A)$ does.

- **aᵢ ∈ ℝⁿ; row i is aᵢᵀ**: In the lecture’s data model, $a_{i}$ is the feature vector for measurement i, so $(A\theta )_{i} = a_{i}^{\mathsf T}\theta  = \theta ^{\mathsf T}a_{i}$. This notation names rows via their transposes. A[:, j] in Python instead selects a column of A, a vector in $\mathbb R^{m}$.

- **θ; θ̂; b̂ = Aθ̂**: $\theta$ is any candidate coefficient vector; the hat marks a minimizing choice $\hat{\theta}$. $\hat{b}$ is the fitted output, not another name for the coefficients. Here $\hat{\theta}$ has 2 coordinates and $\hat{b}$ has 3. The lecture writes the fitted vector as $A\hat{\theta}; \hat{b}$ is an added shorthand on this page.

- **e = b − b̂; f(θ) = ‖Aθ − b‖₂²**: e is the observed-minus-fitted residual vector. At $\hat{\theta}, A\hat{\theta} - b = -e$; both signs have the same squared length. The lecture uses |v| for Euclidean vector length; this page writes $\Vert v\Vert _{2}$. The actual measurement noise $\varepsilon _{i}$ in the model need not equal the fitted residual $e_{i}$.

- **⟨u,v⟩ = uᵀv; e ⊥ Col(A)**: $\mathsf T$ means transpose; $\perp$ means orthogonal. The condition means e has zero inner product with every column combination of A. It is equivalent to $A^{\mathsf T}e=0$, not to $e=0$.

- **G = AᵀA; h = Aᵀb; Gθ̂ = h**: G is an n × n Gram matrix and h has n entries. These names abbreviate the normal equations; G is not the original design matrix A. “Normal” refers to a perpendicular residual, not a normal probability distribution.

- **P = A(AᵀA)⁻¹Aᵀ; b̂ = Pb**: This formula requires independent columns: $\operatorname{rank}(A) = n$. P has shape m × m, acts on outputs, and satisfies $P^{\mathsf T} = P$ and $P^{2} = P$. The map $(A^{\mathsf T}A)^{-1}A^{\mathsf T}$ has shape n × m and computes coefficients; it is not the projection matrix.

- **A = QR; QᵀQ = Iₙ; P = QQᵀ**: For a full-column-rank A, reduced Q has shape m × n and orthonormal columns; R is an invertible n × n upper triangular matrix. For m > n, Q is not a square orthogonal matrix: $Q^{\mathsf T}Q = I_{n}$ but $QQ^{\mathsf T} = P$ generally differs from $I_{m}$.

- **arg min; rank(A); Null(A)**: arg min names the coefficient vectors attaining the smallest objective, not the objective value itself. They form $\hat{\theta} + \operatorname{Null}(A)$. Rank counts independent columns. If $\operatorname{rank}(A) = n$ the coefficient is unique; otherwise the fitted output stays unique while coefficients may differ.

- **NumPy: (3, 2), (2,), .T, @, **2**: A.shape is (3, 2); a 1D $\theta$ array has shape (2,), not (2, 1). .T transposes a matrix, but does not turn a 1D array into a column array. @ is matrix multiplication; * is elementwise multiplication or scalar scaling; **2 squares entries. Python indices start at 0.

- **lstsq: theta_hat, sums_squared, rank_A, singular_values**: np.linalg.lstsq returns four results. sums_squared contains sums of squared residuals, not the residual vector e. It is empty when rank(A) < n or $m \le  n$; empty does not mean zero error. Compute $e = b - A \mathbin{\mathtt{@}} \mathtt{theta\_hat}$ and e @ e directly.

## Key ideas

### Objective and minimizer

$A\theta$ is the prediction for candidate coefficients $\theta$. Choose $\hat{\theta}$ to minimize the sum of squared differences between predictions and observations.

f(θ) = ‖Aθ − b‖₂²; θ̂ ∈ arg min f

Keep in mind: The coefficient $\hat{\theta}$ is a vector; the minimum squared error $f(\hat{\theta})$ is a scalar.

### Normal equations

At a minimum, the residual has zero inner product with every column of A. This gives a smaller system in coefficient space.

AᵀAθ̂ = Aᵀb ⇔ Aᵀe = 0

Keep in mind: These equations hold for every least-squares minimizer. Their inverse formula requires full column rank.

### Fitted values and residual

The fitted vector $\hat{b}$ belongs to the reachable subspace; the remaining part e belongs to its orthogonal complement.

b = b̂ + e; b̂ ∈ Col(A), e ∈ Null(Aᵀ)

Keep in mind: Orthogonality does not mean the residual is zero. Here $\hat{b}$ and e live in $\mathbb R^{3}$, while $\hat{\theta}$ lives in $\mathbb R^{2}$.

### Uniqueness

The nearest reachable output is always unique. Different coefficients can produce that same output if the null space is nontrivial.

all minimizers = θ̂ + Null(A)

Keep in mind: Independent columns give one coefficient vector; dependent columns give a family with the same fitted values.

## 1. Show why the exact system is inconsistent

The normal vector $\ell$ detects an unreachable component of b. The rows are measurements and the columns are feature directions in measurement space.

**Predict first:** If every $A\theta$ has zero inner product with $\ell$, can an output with $\ell ^{\mathsf T}b = 1$ be reached?

- Line 1, `import numpy as np`: Loads NumPy. Shape: module import. Operation: Use np for array creation and linear algebra.

- Line 2, `A = np.array([[-1., 1.], [0., 1.], [1., 0.]])`: Stores the lecture matrix by rows. Shape: 3 rows × 2 features → A: (3, 2). Operation: A @ theta will output $(-\theta _{1} + \theta _{2}, \theta _{2}, \theta _{1})$.

- Line 3, `b = np.array([1., 1., 1.])`: Stores three observed responses. Shape: b: (3,). Operation: All three measurements request the value 1.

- Line 4, `left_normal = np.array([1., -1., 1.])`: Chooses a normal to the reachable plane. Shape: left_normal: (3,). Operation: $\ell  = (1,-1,1)^{\mathsf T}$ is perpendicular to each column of A.

- Line 5, `reachability_check = (A.T @ left_normal, left_normal @ b)`: Checks that b is outside that plane. Shape: (2, 3) @ (3,) → (2,); (3,) @ (3,) → scalar. Operation: $A^{\mathsf T}\ell  = (0,0)^{\mathsf T}$ but $\ell ^{\mathsf T}b = 1$. Every $A\theta$ has $\ell ^{\mathsf T}A\theta  = 0$, so none equals b.

In [ ]:
import numpy as np
A = np.array([[-1., 1.], [0., 1.], [1., 0.]])
b = np.array([1., 1., 1.])
left_normal = np.array([1., -1., 1.])
reachability_check = (A.T @ left_normal, left_normal @ b)

In [ ]:
print("A =", A, "b =", b, "(A.T @ ell, ell @ b) =", reachability_check, sep="\n")

**Read the result:** The reachable plane satisfies $z_{1} - z_{2} + z_{3} = 0$. The observation b does not satisfy that equation.

Aᵀℓ = 0; ℓᵀb = 1 ≠ 0

Least squares asks for the nearest point on this plane, not an exact solution to the inconsistent system.

## 2. Separate the candidate, its output, and its error

The objective depends on the three output errors, not on the size of the two coefficients. A smaller residual length and a smaller squared residual length have the same minimizers.

**Predict first:** Which array has two entries, which has three, and which quantity is a scalar?

- Line 1, `theta_trial = np.array([0., 0.])`: Chooses a simple candidate coefficient. Shape: theta_trial: (2,). Operation: The trial coefficient $\theta  = (0,0)^{\mathsf T}$.

- Line 2, `fitted_trial = A @ theta_trial`: Computes three predictions. Shape: (3, 2) @ (2,) → fitted_trial: (3,). Operation: $A\theta  = (0,0,0)^{\mathsf T}$.

- Line 3, `residual_trial = b - fitted_trial`: Subtracts predictions from observations. Shape: (3,) − (3,) → residual_trial: (3,). Operation: $b - A\theta  = (1,1,1)^{\mathsf T}$.

- Line 4, `sse_trial = residual_trial @ residual_trial`: Adds the squared residual entries. Shape: (3,) @ (3,) → scalar. Operation: $1^{2} + 1^{2} + 1^{2} = 3$; this is the sum of squared errors (SSE).

In [ ]:
theta_trial = np.array([0., 0.])
fitted_trial = A @ theta_trial
residual_trial = b - fitted_trial
sse_trial = residual_trial @ residual_trial

In [ ]:
print("theta_trial =", theta_trial, "fitted_trial =", fitted_trial, "residual_trial =", residual_trial, "SSE =", sse_trial, sep="\n")

**Read the result:** Choosing $\theta  = 0$ reaches the origin. We can reduce the error by moving the fitted output within Col(A).

f(0) = ‖b‖₂² = 3

The objective is a sum, not a mean. Dividing by m would rescale the objective but leave the minimizers unchanged.

## 3. Solve the lecture’s normal equations

A zero gradient gives $A^{\mathsf T}A\hat{\theta} = A^{\mathsf T}b$. The columns are independent here, making G positive definite. solve computes the coefficient without explicitly forming an inverse.

**Predict first:** Why is G a 2 × 2 matrix even though there are three measurements?

- Line 1, `G = A.T @ A`: Forms the Gram matrix from column dot products. Shape: (2, 3) @ (3, 2) → G: (2, 2). Operation: $G = [[2,-1],[-1,2]]$. Its determinant is 3, so it is invertible here.

- Line 2, `h = A.T @ b`: Forms the normal-equation right-hand side. Shape: (2, 3) @ (3,) → h: (2,). Operation: $h = (0,2)^{\mathsf T}$ collects the inner products of b with A’s columns.

- Line 3, `theta_normal = np.linalg.solve(G, h)`: Solves $G\theta  = h$ for the coefficients. Shape: solve((2, 2), (2,)) → theta_normal: (2,). Operation: $2\theta _{1} - \theta _{2} = 0$ and $-\theta _{1} + 2\theta _{2} = 2$ give $\hat{\theta} = (2/3,4/3)^{\mathsf T}$.

In [ ]:
G = A.T @ A
h = A.T @ b
theta_normal = np.linalg.solve(G, h)

In [ ]:
print("G =", G, "h =", h, "theta_normal =", theta_normal, sep="\n")

**Read the result:** The normal equations are a system for coefficients, not three fitted values. Invertibility of G depends on independent columns.

Gθ̂ = h; θ̂ = (2/3,4/3)ᵀ

This step explains the formula for a small full-rank example. General computations later use lstsq directly on A.

## 4. Check the fitted vector and perpendicular residual

The residual need not vanish. Its perpendicularity proves that moving the fit along any column-space direction only adds squared error; the proof below uses Pythagoras.

**Predict first:** Does $A^{\mathsf T}e = 0$ say that every measurement is fitted exactly?

- Line 1, `fitted = A @ theta_normal`: Maps coefficients to the best fitted output. Shape: (3, 2) @ (2,) → fitted: (3,). Operation: $\hat{b} = (-2/3 + 4/3, 4/3, 2/3)^{\mathsf T} = (2/3,4/3,2/3)^{\mathsf T}$.

- Line 2, `residual = b - fitted`: Computes the remaining output error. Shape: (3,) − (3,) → residual: (3,). Operation: $e = (1/3,-1/3,1/3)^{\mathsf T}$. Negative $e_{2}$ means observation 2 is below its prediction.

- Line 3, `orthogonality = A.T @ residual`: Checks perpendicularity to both feature columns. Shape: (2, 3) @ (3,) → orthogonality: (2,). Operation: $A^{\mathsf T}e \approx  (0,0)^{\mathsf T}$ in floating point; exact fractions give zero.

- Line 4, `sse = residual @ residual`: Measures the minimum squared error. Shape: (3,) @ (3,) → scalar. Operation: $(1/3)^{2} + (-1/3)^{2} + (1/3)^{2} = 1/3$, compared with baseline 3.

In [ ]:
fitted = A @ theta_normal
residual = b - fitted
orthogonality = A.T @ residual
sse = residual @ residual

In [ ]:
print("fitted =", fitted, "residual =", residual, "A.T @ residual =", orthogonality, "SSE =", sse, sep="\n")

**Read the result:** The diagram is a 2D orthonormal slice through $\hat{b}$ and e inside $\mathbb R^{3}$. Its axes point along these two perpendicular directions; they are not the measurement coordinates $b_{1}$ and $b_{2}$.

b = b̂ + e; Aᵀe = 0; min f = 1/3

$A^{\mathsf T}e = 0$ is a statement about two inner products, not three residual entries. Exact values are shown; numerical checks use a tolerance.

## 5. See least squares as an orthogonal projection

P acts on a three-entry target and returns a three-entry fitted vector. It does not return the two coefficients. Symmetry gives perpendicular projection; idempotence means an already projected vector stays fixed.

**Predict first:** Why must P have shape (3, 3), rather than (2, 2)?

- Line 1, `P = A @ np.linalg.solve(G, A.T)`: Constructs the projection map in output space. Shape: (3, 2) @ solve((2, 2), (2, 3)) → P: (3, 3). Operation: The solved factor is $G^{-1}A^{\mathsf T}$, so $P = AG^{-1}A^{\mathsf T}$ without forming $G^{-1}$ explicitly.

- Line 2, `projected = P @ b`: Projects the target b. Shape: (3, 3) @ (3,) → projected: (3,). Operation: $Pb = (2/3,4/3,2/3)^{\mathsf T} = \hat{b}$.

- Line 3, `projector_ok = np.allclose(P.T, P) and np.allclose(P @ P, P)`: Checks symmetry and idempotence numerically. Shape: two matrix comparisons → boolean. Operation: $P^{\mathsf T}\approx P$ and $P^2\approx P$; exact algebra gives equality. `projector_ok` is `True`.

In [ ]:
P = A @ np.linalg.solve(G, A.T)
projected = P @ b
projector_ok = np.allclose(P.T, P) and np.allclose(P @ P, P)

In [ ]:
print("P =", P, "P @ b =", projected, "symmetric and idempotent =", projector_ok, sep="\n")

**Read the result:** P projects onto $\operatorname{Col}(A); I_{3} - P$ projects onto its orthogonal complement. Their outputs sum to b.

Pᵀ = P; P² = P; Pb = b̂; (I₃ − P)b = e

This full-rank formula is valid because G is invertible. Projection itself remains defined when the columns are dependent.

## 6. Recover the same fit from reduced QR

Use the lecture’s explicit Q to keep the signs familiar. Orthonormal coordinates reduce least squares to $R\hat{\theta} = Q^{\mathsf T}b$, and the fitted vector is $QQ^{\mathsf T}b$.

**Predict first:** $Q^{\mathsf T}Q$ is $I_{2}$. Does that make $QQ^{\mathsf T}$ equal to $I_{3}$?

- Line 1, `q1 = np.array([-1., 0., 1.]) / np.sqrt(2)`: Normalizes A’s first column. Shape: (3,) ÷ scalar → q1: (3,). Operation: $q_{1} = (-1,0,1)^{\mathsf T}/\sqrt{2}$ has unit length.

- Line 2, `q2 = np.array([1., 2., 1.]) / np.sqrt(6)`: Uses the second lecture Gram–Schmidt direction. Shape: (3,) ÷ scalar → q2: (3,). Operation: $q_{2} = (1,2,1)^{\mathsf T}/\sqrt{6}$ has unit length and is perpendicular to $q_{1}$.

- Line 3, `Q = np.column_stack((q1, q2))`: Stores the orthonormal basis as columns. Shape: two (3,) arrays → Q: (3, 2). Operation: $Q^{\mathsf T}Q = I_{2}$ and $\operatorname{Col}(Q) = \operatorname{Col}(A)$.

- Line 4, `R = Q.T @ A`: Computes the coefficient matrix of A in that basis. Shape: (2, 3) @ (3, 2) → R: (2, 2). Operation: $R = [[\sqrt{2},-1/\sqrt{2}],[0,\sqrt{6}/2]]$; its diagonal is nonzero.

- Line 5, `theta_qr = np.linalg.solve(R, Q.T @ b)`: Solves the smaller triangular system. Shape: solve((2, 2), (2,)) → theta_qr: (2,). Operation: $R\hat{\theta} = Q^{\mathsf T}b$ gives the same $(2/3,4/3)^{\mathsf T}$.

In [ ]:
q1 = np.array([-1., 0., 1.]) / np.sqrt(2)
q2 = np.array([1., 2., 1.]) / np.sqrt(6)
Q = np.column_stack((q1, q2))
R = Q.T @ A
theta_qr = np.linalg.solve(R, Q.T @ b)

In [ ]:
print("Q =", Q, "R =", R, "theta_qr =", theta_qr, sep="\n")

**Read the result:** Q is a rectangular matrix with orthonormal columns. R is a square upper triangular matrix; the two factors reconstruct A.

θ_qr = θ_normal; QQᵀ = P ≠ I₃

A library QR may flip matching signs in Q and R. The product A and the projection $QQ^{\mathsf T}$ remain the same.

## 7. Use lstsq for the computation and interpret all four returns

The lecture’s formulas explain the answer. lstsq computes it without explicitly building $A^{\mathsf T}A$, and also handles dependent columns. Its second return is a summary of squared errors, not a residual vector.

**Predict first:** What is the shape of sums_squared here, and why is it different from e.shape?

- Line 1, `theta_hat, sums_squared, rank_A, singular_values = np.linalg.lstsq(A, b, rcond=None)`: Solves least squares directly from A and b. Shape: (3, 2), (3,) → (2,), (1,), scalar, (2,). Operation: Returns $\hat{\theta}$, a squared-residual summary [1/3], rank 2, and singular values $[\sqrt{3},1]$. rcond=None uses NumPy’s numerical rank cutoff.

- Line 2, `agreement = np.allclose(theta_hat, theta_normal, rtol=0, atol=1e-10)`: Checks agreement with the normal equations. Shape: two (2,) arrays → boolean. Operation: agreement = True within absolute tolerance 1e-10.

- Line 3, `sse_direct = (b - A @ theta_hat) @ (b - A @ theta_hat)`: Computes the squared error from the actual residual. Shape: (3,) @ (3,) → scalar. Operation: $\mathtt{sse\_direct} \approx  1/3$. This remains meaningful even when the summary array is empty.

In [ ]:
theta_hat, sums_squared, rank_A, singular_values = np.linalg.lstsq(A, b, rcond=None)
agreement = np.allclose(theta_hat, theta_normal, rtol=0, atol=1e-10)
sse_direct = (b - A @ theta_hat) @ (b - A @ theta_hat)

In [ ]:
print("theta_hat =", theta_hat, "SSE summary =", sums_squared, "rank =", rank_A, "singular values =", singular_values, "agreement =", agreement, "direct SSE =", sse_direct, sep="\n")

**Read the result:** Normal equations, reduced QR, and lstsq all yield $\hat{\theta} = (2/3,4/3)^{\mathsf T}$ in this example. Each produces the same fitted vector.

θ_normal ≈ θ_qr ≈ θ_hat; ‖b − Aθ̂‖₂² ≈ 1/3

For general data, avoid forming $A^{\mathsf T}A$ just to fit the model; it can worsen conditioning. Direct least-squares solvers are the practical computation path.

## 8. Change coefficients without changing the best output

Dependent columns remove coefficient uniqueness, not fitted-value uniqueness. The two duplicate-column coefficients can trade off freely; only their sum contributes to predictions.

**Predict first:** An empty sums_dep is returned. Does the fit suddenly become exact?

- Line 1, `A_dep = np.column_stack((A, A[:, 1]))`: Duplicates the second feature column. Shape: (3, 2) + (3,) → A_dep: (3, 3). Operation: Column 3 equals column 2, so $\operatorname{rank}(\mathtt{A\_dep}) = 2 < 3$; the output space is unchanged.

- Line 2, `theta_dep, sums_dep, rank_dep, singular_dep = np.linalg.lstsq(A_dep, b, rcond=None)`: Fits with dependent columns. Shape: (3, 3), (3,) → (3,), (0,), scalar, (3,). Operation: lstsq returns the minimum-norm coefficient $(2/3,2/3,2/3)^{\mathsf T}$, rank 2, and an empty sums_dep.

- Line 3, `null_direction = np.array([0., 1., -1.])`: Finds a direction that disappears. Shape: null_direction: (3,). Operation: $\mathtt{A\_dep}\begin{pmatrix}0\\1\\-1\end{pmatrix}=0$: column 2 minus column 3 is zero.

- Line 4, `theta_shifted = theta_dep + null_direction`: Makes a different minimizing coefficient. Shape: (3,) + (3,) → (3,). Operation: $\theta _{\mathrm{shifted}} = (2/3,5/3,-1/3)^{\mathsf T}$. Any real multiple of this null direction could be added.

- Line 5, `fits_dep = np.column_stack((A_dep @ theta_dep, A_dep @ theta_shifted))`: Compares the two fitted outputs. Shape: two (3,) arrays → fits_dep: (3, 2). Operation: Both output columns equal $(2/3,4/3,2/3)^{\mathsf T}$.

- Line 6, `sse_dep = (b - A_dep @ theta_dep) @ (b - A_dep @ theta_dep)`: Computes the error despite the empty summary. Shape: (3,) @ (3,) → scalar. Operation: $\mathtt{sse\_dep} \approx  1/3$, not zero. The same column space gives the same projection.

In [ ]:
A_dep = np.column_stack((A, A[:, 1]))
theta_dep, sums_dep, rank_dep, singular_dep = np.linalg.lstsq(A_dep, b, rcond=None)
null_direction = np.array([0., 1., -1.])
theta_shifted = theta_dep + null_direction
fits_dep = np.column_stack((A_dep @ theta_dep, A_dep @ theta_shifted))
sse_dep = (b - A_dep @ theta_dep) @ (b - A_dep @ theta_dep)

In [ ]:
print("theta_dep =", theta_dep, "theta_shifted =", theta_shifted, "fitted outputs =", fits_dep, "SSE summary =", sums_dep, "rank =", rank_dep, "singular values =", singular_dep, "direct SSE =", sse_dep, sep="\n")

**Read the result:** The three-entry coefficients differ, but their three-entry fitted vectors match. The minimum squared error remains 1/3.

all minimizers = θ_dep + t(0,1,−1)ᵀ, t ∈ ℝ

Here $\operatorname{Null}(\mathtt{A\_dep}) = \operatorname{span}\{(0,1,-1)^{\mathsf T}\}$. lstsq chooses the smallest coefficient norm among this family; ordinary least squares minimizes the residual norm.

## Why least squares is projection

### An unreachable target still has a nearest reachable output

The outputs $A\theta$ fill Col(A), a finite-dimensional subspace of $\mathbb R^{m}$. Every b has a unique orthogonal projection onto that subspace. Because this projection belongs to Col(A), some coefficient vector produces it. Thus least squares always has a minimizer, even when the exact equation has none.

For the running example, $\ell  = (1,-1,1)^{\mathsf T}$ satisfies $A^{\mathsf T}\ell  = 0$. Every reachable output obeys $\ell ^{\mathsf T}A\theta  = 0$, but $b = (1,1,1)^{\mathsf T}$ gives $\ell ^{\mathsf T}b = 1$. This certifies $b \notin  \operatorname{Col}(A)$. The obstruction is a nonzero component perpendicular to the reachable plane.

θ̂ minimizes ‖Aθ − b‖₂²; b̂ = Aθ̂ is the closest point of Col(A) to b

### The normal equations follow from the objective

Expand $f(\theta ) = (A\theta  - b)^{\mathsf T}(A\theta  - b) = \theta ^{\mathsf T}A^{\mathsf T}A\theta  - 2b^{\mathsf T}A\theta  + b^{\mathsf T}b$. $A^{\mathsf T}A$ is symmetric, so the gradient is $\nabla f(\theta ) = 2A^{\mathsf T}A\theta  - 2A^{\mathsf T}b$. At a minimum, the gradient is zero, giving $A^{\mathsf T}A\hat{\theta} = A^{\mathsf T}b$.

This condition is sufficient as well as necessary: the Hessian is $2A^{\mathsf T}A$ and $z^{\mathsf T}A^{\mathsf T}Az = \Vert Az\Vert _{2}^{2} \ge  0$. The objective is convex, so a stationary point is a global minimizer. When the columns are independent, this squared length is positive for every nonzero z; the objective is strictly convex and the coefficient is unique. The geometric proof below establishes sufficiency without calculus.

∇f(θ̂) = 0 ⇔ Aᵀ(Aθ̂ − b) = 0 ⇔ Aᵀe = 0

### A perpendicular residual proves global optimality

Suppose $A^{\mathsf T}e = 0$ with $e = b - A\hat{\theta}$. For any candidate $\theta , b - A\theta  = e - A(\theta  - \hat{\theta})$. The two terms are perpendicular: e is orthogonal to Col(A), while $A(\theta  - \hat{\theta})$ belongs to that subspace. Pythagoras therefore splits the squared error into the best error plus a nonnegative extra term.

Equality holds precisely when $A(\theta  - \hat{\theta}) = 0$. This proves both optimality and the full minimizer set $\hat{\theta} + \operatorname{Null}(A)$. For our example, $e = (1/3,-1/3,1/3)^{\mathsf T}$, so the minimum squared error is 1/3. The minimum residual length is $\sqrt{1/3}$; these are different numbers. The coefficient norm $\Vert \hat{\theta}\Vert _{2}$ is a third quantity and is not the ordinary least-squares objective.

‖b − Aθ‖₂² = ‖e‖₂² + ‖A(θ − θ̂)‖₂² ≥ ‖e‖₂²

### The inverse formula has a rank assumption

For $z \in  \mathbb R^{n}, z^{\mathsf T}Gz = \Vert Az\Vert _{2}^{2}$. Thus $G = A^{\mathsf T}A$ is invertible exactly when $\operatorname{Null}(A) = \{0\}$, equivalently $\operatorname{rank}(A) = n$. Under this assumption $\hat{\theta} = (A^{\mathsf T}A)^{-1}A^{\mathsf T}b$ and $\hat{b} = Pb$, with $P = A(A^{\mathsf T}A)^{-1}A^{\mathsf T}$.

The normal equations remain valid when columns are dependent, but the displayed inverse does not exist. Projection onto Col(A) still exists and is unique. One can use an orthonormal basis $Q_{r}$ of its r-dimensional column space to obtain $P = Q_{r}Q_{r}^{\mathsf T}$, or write $P = AA^{+}$ using the Moore–Penrose pseudoinverse $A^{+}$. The rank-deficient code example avoids using the invalid inverse formula.

rank(A) = n ⇒ unique θ̂; rank(A) < n ⇒ multiple θ̂, one fitted output Aθ̂

### QR computes the same projection with orthonormal coordinates

For full column rank, $A = QR$ with reduced Q having orthonormal columns. Split b into $QQ^{\mathsf T}b$ and its perpendicular residual. Then $\Vert A\theta  - b\Vert _{2}^{2} = \Vert R\theta  - Q^{\mathsf T}b\Vert _{2}^{2} + \Vert (I - QQ^{\mathsf T})b\Vert _{2}^{2}$. The second term cannot be changed by $\theta$; the first becomes zero when $R\hat{\theta} = Q^{\mathsf T}b$.

This gives the same fitted output $QQ^{\mathsf T}b$ as the Gram-matrix formula. In this example Q is 3 × 2, so $Q^{\mathsf T}Q = I_{2}$, while $QQ^{\mathsf T}$ is a rank-two projection in $\mathbb R^{3}$. Different sign choices for columns of Q and matching rows of R leave QR and $QQ^{\mathsf T}$ unchanged.

Rθ̂ = Qᵀb; Aθ̂ = QRθ̂ = QQᵀb = Pb

### Distinguish a linear subspace from a shifted tangent plane

Printed slide 33 projects $s = (2,3,0)^{\mathsf T}$ onto the tangent plane through $p_{0} = (1,2,-1)^{\mathsf T}$. That plane is $p_{0} + \operatorname{Col}(A)$. First use the displacement $b = s - p_{0} = (1,1,1)^{\mathsf T}$, which is exactly the target in our least-squares example.

$Pb = (2/3,4/3,2/3)^{\mathsf T}$ is the projected displacement. To recover the projected point in the original coordinates, add $p_{0}: \hat{s} = p_{0} + Pb = (5/3,10/3,-1/3)^{\mathsf T}$. The projection onto the shifted plane is an affine map of s; P itself projects onto the subspace through the origin.

ŝ = p₀ + P(s − p₀); s − ŝ = b − Pb = e

### Use the formulas to understand the problem and a solver to compute it

The Gram-matrix step exposes the lecture’s normal equations and uses np.linalg.solve(G, h) rather than explicitly computing $G^{-1}$. Forming $A^{\mathsf T}A$ can worsen conditioning: for full column rank, its spectral condition number is the square of A’s. Solving with G instead of inverting it does not remove that effect. The small well-conditioned example is suitable for inspecting the formula.

For general data, use np.linalg.lstsq(A, b, rcond=None), which works directly with A and supports rank-deficient systems. It returns a minimum-Euclidean-norm coefficient if minimizers are not unique. The rank cutoff depends on singular values and numerical precision, so near-dependent columns require attention to scale and tolerance. An approximately zero $A^{\mathsf T}e$ checks stationarity to the chosen tolerance; the algebra above proves optimality.



## Numerical checks

The arguments above explain existence, uniqueness of the fit, and the full family of coefficients. These numerical checks verify the displayed example. Floating-point comparisons use an explicit absolute tolerance.

In [ ]:
def close(actual, expected):
    return np.allclose(actual, expected, rtol=0, atol=1e-10)

assert A.shape == (3, 2) and b.shape == (3,)
assert close(A.T @ left_normal, np.zeros(2)) and close(left_normal @ b, 1)
assert close(G, [[2, -1], [-1, 2]]) and close(h, [0, 2])
assert close(theta_normal, [2 / 3, 4 / 3])
assert close(theta_qr, theta_normal) and close(theta_hat, theta_normal)
assert close(fitted, [2 / 3, 4 / 3, 2 / 3])
assert close(residual, [1 / 3, -1 / 3, 1 / 3])
assert close(A.T @ residual, np.zeros(2))
assert close(sse, 1 / 3) and close(sse_direct, 1 / 3)
assert close(P, np.array([[2, 1, -1], [1, 2, 1], [-1, 1, 2]]) / 3)
assert close(P.T, P) and close(P @ P, P)
assert close(P @ b, fitted) and close((np.eye(3) - P) @ b, residual)
assert close(Q.T @ Q, np.eye(2)) and close(Q @ R, A)
assert close(Q @ Q.T, P) and close(np.tril(R, -1), np.zeros((2, 2)))
assert rank_A == 2 and close(singular_values, [np.sqrt(3), 1])
assert sums_squared.shape == (1,) and close(sums_squared, [1 / 3])
assert A_dep.shape == (3, 3) and rank_dep == 2
assert sums_dep.shape == (0,) and close(sse_dep, 1 / 3)
assert close(theta_dep, [2 / 3, 2 / 3, 2 / 3])
assert close(theta_shifted, [2 / 3, 5 / 3, -1 / 3])
assert close(A_dep @ null_direction, np.zeros(3))
assert close(fits_dep, np.column_stack((fitted, fitted)))
for t in [-3., -0.5, 0., 1., 4.]:
    candidate = theta_dep + t * null_direction
    assert close(A_dep @ candidate, fitted)
    assert close(candidate @ candidate, theta_dep @ theta_dep + 2 * t ** 2)
print("All example checks passed.")

## Use NumPy’s reduced QR

A library may choose different column signs from the lecture. Check the reconstructed matrix, projection, and coefficients rather than expecting identical entries in Q and R. This solve assumes full column rank.

In [ ]:
Q_library, R_library = np.linalg.qr(A, mode="reduced")
theta_library = np.linalg.solve(R_library, Q_library.T @ b)
assert Q_library.shape == (3, 2) and R_library.shape == (2, 2)
assert close(Q_library @ R_library, A)
assert close(Q_library @ Q_library.T, P)
assert close(theta_library, theta_hat)
print("Q_library =", Q_library, "R_library =", R_library, "theta_library =", theta_library, sep="\n")

## Project onto the lecture’s tangent plane

The lecture first subtracts the base point p₀ = (1,2,−1)ᵀ from s = (2,3,0)ᵀ. P projects that displacement onto Col(A). Add p₀ back to obtain a point on the affine plane p₀ + Col(A).

In [ ]:
p0 = np.array([1., 2., -1.])
s = np.array([2., 3., 0.])
displacement = s - p0
projected_point = p0 + P @ displacement
assert close(displacement, b)
assert close(projected_point, [5 / 3, 10 / 3, -1 / 3])
assert close(A.T @ (s - projected_point), np.zeros(2))
print("projected displacement =", P @ displacement, "projected point =", projected_point, sep="\n")

## Try another target and check Pythagoras

Change b_new and a candidate coefficient. Even for another target, the minimizing residual is perpendicular to all reachable directions. The squared error of a candidate is the minimum squared error plus the squared distance between its fitted output and the optimal fitted output.

In [ ]:
b_new = np.array([2., -1., 0.5])
theta_new, _, rank_new, _ = np.linalg.lstsq(A, b_new, rcond=None)
fit_new = A @ theta_new
e_new = b_new - fit_new
candidate = np.array([-1., 2.])
error_candidate = b_new - A @ candidate
fit_difference = A @ (candidate - theta_new)
assert rank_new == 2 and close(A.T @ e_new, np.zeros(2))
assert close(P @ b_new, fit_new)
assert close(error_candidate @ error_candidate,
             e_new @ e_new + fit_difference @ fit_difference)
print("new coefficients =", theta_new, "minimum SSE =", e_new @ e_new,
      "candidate SSE =", error_candidate @ error_candidate, sep="\n")

## Check your understanding

**If $A^{\mathsf T}e = 0$, must every residual entry be zero?**

<details><summary>Show explanation</summary>

No. It means the residual is perpendicular to the column space. Here $e = (1/3,-1/3,1/3)^{\mathsf T}$ is nonzero, yet $A^{\mathsf T}e = 0$. Zero residual occurs exactly when b is reachable.

</details>

**What lives in $\mathbb R^{2}$, and what lives in $\mathbb R^{3}$?**

<details><summary>Show explanation</summary>

$\theta$ and $\hat{\theta}$ have two coefficient coordinates. $b, \hat{b} = A\hat{\theta}$, and e have three measurement coordinates. P acts in $\mathbb R^{3}$; the coefficient map $(A^{\mathsf T}A)^{-1}A^{\mathsf T}$ maps $\mathbb R^{3}$ to $\mathbb R^{2}$.

</details>

**Why minimize squared distance rather than distance?**

<details><summary>Show explanation</summary>

For a nonnegative length, squaring is strictly increasing, so both objectives have the same minimizers. Their minimum values differ: the squared error is 1/3 here, while the residual length is $1/\sqrt{3}$.

</details>

**Does least squares always give unique coefficients?**

<details><summary>Show explanation</summary>

It always gives a unique fitted output, the projection of b onto Col(A). Coefficients are unique exactly when A has independent columns. With dependent columns, adding any z in Null(A) preserves the output and error.

</details>

**Can we use $(A^{\mathsf T}A)^{-1}$ if a column is duplicated?**

<details><summary>Show explanation</summary>

No: duplicate columns make $A^{\mathsf T}A$ singular. The normal equations still characterize minimizers. Use lstsq, an orthonormal basis of Col(A), or a pseudoinverse.

</details>

**What does an empty sums_squared array from lstsq mean?**

<details><summary>Show explanation</summary>

It means that this summary is not returned for the matrix shape or rank; it does not mean a perfect fit. Our 3 × 3 rank-two example returns an empty array and still has squared error 1/3. Compute e @ e.

</details>

**Is Pb the final point on the tangent plane in printed slide 33?**

<details><summary>Show explanation</summary>

Pb is a displacement from $p_{0}$. The final point is $p_{0} + Pb = (5/3,10/3,-1/3)^{\mathsf T}$. This distinction connects the linear projection to a plane that does not pass through the origin.

</details>

## Code references

- [NumPy: lstsq results and rank cutoff](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html)
- [NumPy: reduced QR shapes](https://numpy.org/doc/stable/reference/generated/numpy.linalg.qr.html)